# Módulo 7 · Clase 8 — Clustering

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Aprendizaje no supervisado: qué es y cuándo usarlo |
| 2 | K-Means: algoritmo paso a paso |
| 3 | Elbow method: elegir el número de clusters |
| 4 | Silhouette score: calidad del clustering |
| 5 | Caso real: segmentación de clientes |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.datasets import make_blobs

np.random.seed(42)
print("Librerías listas")

---
## 1. Aprendizaje no supervisado

```
Supervisado:     datos tienen etiquetas (y)
                 objetivo: predecir y para datos nuevos

No supervisado:  datos SIN etiquetas
                 objetivo: descubrir ESTRUCTURA en los datos

Casos de uso del clustering:
  - Segmentación de clientes (marketing)
  - Agrupación de documentos / noticias
  - Detección de anomalías
  - Compresión de imágenes
  - Bioinformática: grupos de genes con comportamiento similar

Algoritmos más comunes:
  K-Means           → grupos esféricos, tamaño similar
  DBSCAN            → grupos de forma arbitraria, outliers explícitos
  Jerárquico        → dendrograma, no requiere K
  Gaussian Mixture  → grupos solapados con probabilidades

Hoy: K-Means
```

---
## 2. K-Means: algoritmo

```
Parámetro requerido: K (número de clusters)

Algoritmo:
  1. Inicializar K centroides al azar
  2. Asignar cada punto al centroide más cercano (distancia euclidiana)
  3. Recalcular los centroides como la media de los puntos asignados
  4. Repetir 2-3 hasta convergencia (centroides no se mueven más)

Criterio de calidad: inercia (dentro del cluster)
  Inercia = Σ dist²(punto, centroide_asignado)
  Menor inercia = clusters más compactos

Limitaciones:
  - Requiere especificar K de antemano
  - Sensible a outliers
  - Asume clusters esféricos y de tamaño similar
  - El resultado depende de la inicialización (usa k-means++ por defecto)
```

In [ ]:
# Visualizar el proceso de K-Means paso a paso
np.random.seed(42)
X_demo, y_true = make_blobs(n_samples=120, centers=3, cluster_std=1.2, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
iteraciones = [1, 2, 4, 10]
colores_cl  = ['tomato', 'steelblue', 'seagreen']

for ax, n_iter in zip(axes, iteraciones):
    km = KMeans(n_clusters=3, max_iter=n_iter, n_init=1,
                init='random', random_state=0)
    labels = km.fit_predict(X_demo)
    for cl in range(3):
        mask = labels == cl
        ax.scatter(X_demo[mask, 0], X_demo[mask, 1],
                   c=colores_cl[cl], s=25, alpha=0.7)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', s=200, marker='X', zorder=10)
    ax.set_title(f'Iteración {n_iter}\nInercia={km.inertia_:.1f}')

plt.suptitle('K-Means: convergencia iterativa', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Uso básico de KMeans en sklearn
km = KMeans(
    n_clusters=3,    # número de clusters K
    n_init=10,       # reiniciar 10 veces con distintos centroides, quedarse con el mejor
    max_iter=300,    # máximo de iteraciones
    random_state=42  # reproducibilidad
)
labels = km.fit_predict(X_demo)  # fit + asignar etiquetas en un solo paso

print(f"Etiquetas únicas: {np.unique(labels)}")
print(f"Tamaño de clusters: {np.bincount(labels)}")
print(f"Inercia: {km.inertia_:.2f}")
print(f"Centroides:\n{km.cluster_centers_.round(2)}")
print(f"Iteraciones hasta convergencia: {km.n_iter_}")

# Visualizar resultado final
fig, ax = plt.subplots(figsize=(6, 4))
for cl in range(3):
    mask = labels == cl
    ax.scatter(X_demo[mask, 0], X_demo[mask, 1],
               label=f'Cluster {cl}', s=30, alpha=0.8)
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
           c='black', s=300, marker='X', label='Centroides', zorder=10)
ax.set_title('K-Means final (K=3)')
ax.legend(); plt.tight_layout(); plt.show()

---
## 3. Elbow Method: elegir K

In [ ]:
# El truco: la inercia siempre baja con más clusters
# Buscar el "codo" donde la mejora empieza a ser marginal

np.random.seed(42)
X_cl, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.5, random_state=42)

ks      = range(1, 12)
inercias = []
for k in ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42)
    km_k.fit(X_cl)
    inercias.append(km_k.inertia_)

# Calcular la segunda derivada para detectar el codo automáticamente
deltas  = np.diff(inercias)
deltas2 = np.diff(deltas)
k_codo  = int(np.argmax(np.abs(deltas2)) + 2)  # +2 por los dos diff

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(ks, inercias, 'bo-', ms=7, lw=2)
axes[0].axvline(k_codo, color='red', ls='--', lw=2, label=f'Codo en K={k_codo}')
axes[0].set_xlabel('K (número de clusters)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Elbow Method')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Reducción relativa de inercia
reduccion = [-deltas[i] / inercias[i] * 100 for i in range(len(deltas))]
axes[1].bar(range(2, len(ks)+1), reduccion, color='steelblue')
axes[1].axvline(k_codo, color='red', ls='--', lw=2, label=f'K óptimo={k_codo}')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Reducción de inercia (%)')
axes[1].set_title('Reducción relativa al añadir cada cluster')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"K óptimo sugerido por el elbow: {k_codo} (datos reales tenían 4 clusters)")

---
## 4. Silhouette Score

```
Silhouette de un punto i:
  a(i) = distancia media a puntos del MISMO cluster
  b(i) = distancia media al cluster MÁS CERCANO (distinto)
  s(i) = (b(i) - a(i)) / max(a(i), b(i))

Interpretación:
  s(i) cerca de +1 → punto bien asignado
  s(i) cerca de 0  → punto en la frontera entre clusters
  s(i) < 0         → punto probablemente mal asignado

Silhouette Score global = media de todos los s(i)
  Rango: [-1, 1]. Buscar el K que maximice este valor.
```

In [ ]:
# Silhouette score para distintos K
silhouettes = []
for k in range(2, 10):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels_k = km_k.fit_predict(X_cl)
    sc = silhouette_score(X_cl, labels_k)
    silhouettes.append(sc)
    print(f"K={k}: silhouette={sc:.4f}, inercia={km_k.inertia_:.1f}")

k_opt = range(2, 10)[np.argmax(silhouettes)]
print(f"\nK óptimo por silhouette: {k_opt}")

# Comparar elbow vs silhouette
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(2, 10), silhouettes, 'rs-', ms=7, lw=2)
ax.axvline(k_opt, color='red', ls='--', lw=2, label=f'K óptimo={k_opt}')
ax.set_xlabel('K'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs K')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Gráfico de silhouette por cluster
def plot_silhouette(X, k):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    sc_global = silhouette_score(X, labels)
    sc_muestra = silhouette_samples(X, labels)

    fig, ax = plt.subplots(figsize=(7, 5))
    y_lower = 10
    colores = cm.Set2(np.linspace(0, 0.8, k))
    for cl in range(k):
        sc_cl = np.sort(sc_muestra[labels == cl])
        size_cl = len(sc_cl)
        y_upper = y_lower + size_cl
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sc_cl,
                         facecolor=colores[cl], alpha=0.8)
        ax.text(-0.05, y_lower + size_cl/2, str(cl), fontsize=9)
        y_lower = y_upper + 5

    ax.axvline(sc_global, color='red', ls='--', lw=2)
    ax.set_title(f'Silhouette Plot (K={k}, score global={sc_global:.3f})')
    ax.set_xlabel('Silhouette individual')
    ax.set_ylabel('Cluster')
    ax.set_yticks([]); ax.set_xlim([-0.2, 1.0])
    plt.tight_layout(); plt.show()

# Ver K=3 y K=4 (el correcto)
for k in [3, 4]:
    plot_silhouette(X_cl, k)

---
## 5. Caso real: segmentación de clientes

In [ ]:
# Dataset sintético de clientes de un e-commerce
np.random.seed(42)
n = 400
# Perfil A: clientes frecuentes y de alto valor
A_gasto  = np.random.normal(500, 80, n//4)
A_visitas = np.random.normal(20, 3, n//4)
A_edad   = np.random.normal(35, 5, n//4)
# Perfil B: clientes ocasionales y precio-sensibles
B_gasto  = np.random.normal(100, 30, n//4)
B_visitas = np.random.normal(4, 1.5, n//4)
B_edad   = np.random.normal(25, 4, n//4)
# Perfil C: clientes maduros y estables
C_gasto  = np.random.normal(300, 50, n//4)
C_visitas = np.random.normal(10, 2, n//4)
C_edad   = np.random.normal(55, 8, n//4)
# Perfil D: clientes corporativos
D_gasto  = np.random.normal(1200, 200, n//4)
D_visitas = np.random.normal(8, 2, n//4)
D_edad   = np.random.normal(45, 6, n//4)

df_clientes = pd.DataFrame({
    'gasto_mensual': np.concatenate([A_gasto, B_gasto, C_gasto, D_gasto]),
    'visitas_mes':   np.concatenate([A_visitas, B_visitas, C_visitas, D_visitas]),
    'edad':          np.concatenate([A_edad, B_edad, C_edad, D_edad]),
})
df_clientes = df_clientes.clip(lower=0)  # sin negativos

print("Dataset de clientes:")
print(df_clientes.describe().round(1))

In [ ]:
# 1. Escalar los datos (SIEMPRE antes de clustering)
scaler = StandardScaler()
X_cl_sc = scaler.fit_transform(df_clientes)

# 2. Elbow + Silhouette para elegir K
ks_rango = range(2, 9)
inercias_cl, siluetas_cl = [], []
for k in ks_rango:
    km_k = KMeans(k, n_init=15, random_state=42)
    lab  = km_k.fit_predict(X_cl_sc)
    inercias_cl.append(km_k.inertia_)
    siluetas_cl.append(silhouette_score(X_cl_sc, lab))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks_rango, inercias_cl, 'bo-', ms=7, lw=2)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia')
axes[0].set_title('Elbow Method'); axes[0].grid(True, alpha=0.3)

k_opt_sil = list(ks_rango)[np.argmax(siluetas_cl)]
axes[1].plot(ks_rango, siluetas_cl, 'rs-', ms=7, lw=2)
axes[1].axvline(k_opt_sil, color='red', ls='--', label=f'K óptimo={k_opt_sil}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Clientes: elegir número de segmentos', fontsize=11)
plt.tight_layout(); plt.show()
print(f"K óptimo por silhouette: {k_opt_sil}")

In [ ]:
# 3. Entrenar KMeans con K óptimo
K_FINAL = k_opt_sil
km_final = KMeans(K_FINAL, n_init=20, random_state=42)
df_clientes['segmento'] = km_final.fit_predict(X_cl_sc)

# 4. Caracterizar cada segmento
perfil = df_clientes.groupby('segmento').agg(
    n_clientes     = ('gasto_mensual', 'count'),
    gasto_medio    = ('gasto_mensual', 'mean'),
    visitas_media  = ('visitas_mes', 'mean'),
    edad_media     = ('edad', 'mean'),
).round(1)
print("Perfil de segmentos:")
print(perfil)

# 5. Poner nombres comerciales
orden_gasto = perfil['gasto_medio'].rank(ascending=False)
nombres = {}
for seg in perfil.index:
    rk = int(orden_gasto[seg])
    if rk == 1:   nombres[seg] = 'Premium'
    elif rk == 2: nombres[seg] = 'Frecuente'
    elif rk == 3: nombres[seg] = 'Estable'
    else:         nombres[seg] = 'Ocasional'

df_clientes['nombre_segmento'] = df_clientes['segmento'].map(nombres)
print(f"\nEtiquetas asignadas: {nombres}")

# 6. Visualizar
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_seg = ['#e74c3c','#3498db','#2ecc71','#9b59b6']
for seg in range(K_FINAL):
    mask = df_clientes['segmento'] == seg
    axes[0].scatter(df_clientes.loc[mask,'gasto_mensual'],
                    df_clientes.loc[mask,'visitas_mes'],
                    c=colores_seg[seg], s=25, alpha=0.7,
                    label=nombres[seg])
axes[0].set_xlabel('Gasto mensual (€)')
axes[0].set_ylabel('Visitas al mes')
axes[0].set_title('Segmentación de clientes')
axes[0].legend()

n_segs = df_clientes.groupby('nombre_segmento').size()
axes[1].pie(n_segs, labels=n_segs.index, autopct='%1.1f%%',
            colors=colores_seg[:len(n_segs)],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Distribución de segmentos')
plt.tight_layout(); plt.show()

---
## Práctica

### Ejercicio 1 — Clustering en datos reales

**Enunciado:** Usa el dataset de iris sin las etiquetas:

```python
from sklearn.datasets import load_iris
iris = load_iris()
X = iris.data  # 4 features: sépalo y pétalo
y_real = iris.target  # NO usar para entrenar KMeans
```

1. Aplica Elbow Method y Silhouette Score para K de 2 a 8
2. Entrena KMeans con el K óptimo
3. Compara los clusters encontrados con las etiquetas reales (confusion matrix)
4. ¿Qué porcentaje de muestras clasificó correctamente (aprox.)?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, confusion_matrix

iris = load_iris()
X, y_real = iris.data, iris.target

scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

# Elbow + Silhouette
ks = range(2, 9)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(k, n_init=15, random_state=42)
    lab = km.fit_predict(X_sc)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_sc, lab))

k_opt = list(ks)[np.argmax(siluetas)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, inercias, 'bo-', ms=7); axes[0].set_title('Elbow'); axes[0].set_xlabel('K')
axes[1].plot(ks, siluetas, 'rs-', ms=7); axes[1].set_title('Silhouette'); axes[1].set_xlabel('K')
axes[1].axvline(k_opt, color='red', ls='--', label=f'K={k_opt}')
axes[1].legend(); plt.tight_layout(); plt.show()
print(f"K óptimo: {k_opt}")

# KMeans final
km_final = KMeans(k_opt, n_init=20, random_state=42)
labels = km_final.fit_predict(X_sc)

# Comparar con etiquetas reales
# Alinear labels (pueden estar en distinto orden)
from scipy.optimize import linear_sum_assignment
cm = confusion_matrix(y_real, labels)
row_ind, col_ind = linear_sum_assignment(-cm)
acc_aprox = cm[row_ind, col_ind].sum() / len(y_real)
print(f"\nMatrix de confusión (cluster vs clase real):")
print(cm)
print(f"\nAccuracy aproximada (alineando clusters): {acc_aprox:.2%}")
print("Iris-versicolor e iris-virginica se solapan: el clustering lo refleja.")
```
</details>

---
### Ejercicio 2 — Efecto del escalado

**Enunciado:** Genera datos con features en escalas muy distintas:

```python
np.random.seed(42)
X_sin_escala = np.column_stack([
    np.random.randn(200) * 1000,   # feature A: rango 0–3000
    np.random.randn(200)           # feature B: rango -3 a 3
])
```

1. Aplica KMeans (K=3) SIN escalar y visualiza los clusters
2. Aplica KMeans (K=3) CON StandardScaler y visualiza
3. ¿Cuál tiene mayor silhouette score?
4. Explica por qué el escalado importa en K-Means

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

np.random.seed(42)
X_sin = np.column_stack([
    np.random.randn(200) * 1000,
    np.random.randn(200)
])

X_con = StandardScaler().fit_transform(X_sin)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, X_uso, titulo in zip(axes, [X_sin, X_con], ['Sin escalar', 'Con StandardScaler']):
    km = KMeans(3, n_init=15, random_state=42)
    labels = km.fit_predict(X_uso)
    sc = silhouette_score(X_uso, labels)
    for cl in range(3):
        mask = labels == cl
        ax.scatter(X_uso[mask, 0], X_uso[mask, 1], s=20, alpha=0.7)
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', marker='X', s=200, zorder=10)
    ax.set_title(f'{titulo}\nSilhouette={sc:.4f}')
    ax.set_xlabel('Feature A'); ax.set_ylabel('Feature B')

plt.suptitle('Importancia del escalado en K-Means', fontsize=11)
plt.tight_layout(); plt.show()
print("SIN escalar: la distancia euclidiana la domina feature A (escala 1000x).")
print("K-Means ignora feature B → clusters son bandas verticales.")
print("CON escalado: ambas features contribuyen igual a la distancia.")
```
</details>

---
### Ejercicio 3 — Segmentación de noticias por tópico

**Enunciado:** Usa el dataset de dígitos escritos a mano como si fuera datos sin etiqueta:

```python
from sklearn.datasets import load_digits
digits = load_digits()
X = digits.data   # 1797 imágenes de 64 pixels
y = digits.target # NO usar para entrenar
```

1. Reduce a 2D con PCA antes de clusterizar (para visualización y eficiencia):
   ```python
   from sklearn.decomposition import PCA
   X_2d = PCA(n_components=2, random_state=42).fit_transform(X)
   ```
2. Aplica KMeans con K=10 (el número real de dígitos)
3. Visualiza los clusters en 2D
4. Muestra el silhouette score y comenta el resultado

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target

# Escalar + reducir dimensión
X_sc = StandardScaler().fit_transform(X)
X_2d = PCA(n_components=2, random_state=42).fit_transform(X_sc)

# KMeans con K=10
km = KMeans(10, n_init=20, random_state=42)
labels = km.fit_predict(X_2d)
sc = silhouette_score(X_2d, labels)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
scatter = axes[0].scatter(X_2d[:,0], X_2d[:,1], c=labels, cmap='tab10', s=10, alpha=0.7)
axes[0].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
                c='black', marker='X', s=200, zorder=10)
axes[0].set_title(f'K-Means K=10 (silhouette={sc:.3f})')
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Etiquetas reales para comparación
scatter2 = axes[1].scatter(X_2d[:,0], X_2d[:,1], c=y, cmap='tab10', s=10, alpha=0.7)
axes[1].set_title('Clases reales (dígitos 0-9)')
plt.colorbar(scatter2, ax=axes[1], label='Dígito')

plt.suptitle('Clustering de dígitos en 2D (PCA)', fontsize=11)
plt.tight_layout(); plt.show()

print(f"Silhouette Score con K=10: {sc:.4f}")
print("Comentario: silhouette bajo porque en 2D los dígitos se solapan mucho.")
print("Con más componentes PCA (ej. 20), el clustering mejoraría.")
```
</details>

---
### Ejercicio 4 — Sensibilidad a la inicialización

**Enunciado:** Demuestra que K-Means puede converger a distintos mínimos locales:

1. Usa `make_blobs(n_samples=200, centers=4, cluster_std=2, random_state=42)`
2. Ejecuta KMeans 20 veces con `n_init=1` y `init='random'`, semilla distinta cada vez
3. Guarda la inercia y el silhouette de cada ejecución
4. Compara con una ejecución usando `n_init=20` y `init='k-means++'`
5. ¿Por qué n_init>1 y k-means++ son buenas prácticas?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=200, centers=4, cluster_std=2, random_state=42)

inercias_random, siluetas_random = [], []
for seed in range(20):
    km = KMeans(4, n_init=1, init='random', random_state=seed)
    lab = km.fit_predict(X)
    inercias_random.append(km.inertia_)
    siluetas_random.append(silhouette_score(X, lab))

# K-Means++ con múltiples reintentos
km_plus = KMeans(4, n_init=20, init='k-means++', random_state=42)
lab_plus = km_plus.fit_predict(X)
sc_plus = silhouette_score(X, lab_plus)

print(f"Inicialización random (20 seeds):")
print(f"  Inercia: min={min(inercias_random):.1f}, max={max(inercias_random):.1f}, "
      f"rango={max(inercias_random)-min(inercias_random):.1f}")
print(f"  Silhouette: min={min(siluetas_random):.4f}, max={max(siluetas_random):.4f}")
print(f"\nK-Means++ (n_init=20):")
print(f"  Inercia: {km_plus.inertia_:.1f}")
print(f"  Silhouette: {sc_plus:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(inercias_random, bins=8, color='coral', edgecolor='white')
axes[0].axvline(km_plus.inertia_, color='green', lw=2.5, label='k-means++')
axes[0].set_title('Variabilidad de inercia (init random)'); axes[0].legend()

axes[1].hist(siluetas_random, bins=8, color='steelblue', edgecolor='white')
axes[1].axvline(sc_plus, color='green', lw=2.5, label='k-means++')
axes[1].set_title('Variabilidad de silhouette'); axes[1].legend()

plt.tight_layout(); plt.show()
print("\nConclusion: k-means++ selecciona centroides iniciales bien separados.")
print("n_init>1 garantiza que no nos quedamos en un mínimo local.")
```
</details>

---
### Ejercicio 5 (reto) — Pipeline de segmentación completo

**Enunciado:** Construye una función `segmentar_clientes(df, features, k_max=10)` que:

1. Escala las features
2. Determina automáticamente el K óptimo (elbow + silhouette)
3. Entrena K-Means con el K elegido
4. Añade una columna `'segmento'` al DataFrame
5. Devuelve el DataFrame con segmentos y una tabla resumen por segmento
6. Plotea los segmentos en 2D (usa las 2 primeras features)

Pruébala con el dataset de clientes de esta clase.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

def segmentar_clientes(df, features, k_max=10):
    X = df[features].values
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)

    # Determinar K óptimo
    ks = range(2, k_max+1)
    siluetas = []
    inercias = []
    for k in ks:
        km = KMeans(k, n_init=15, random_state=42)
        lab = km.fit_predict(X_sc)
        siluetas.append(silhouette_score(X_sc, lab))
        inercias.append(km.inertia_)

    k_opt = list(ks)[np.argmax(siluetas)]

    # Entrenar KMeans final
    km_final = KMeans(k_opt, n_init=20, random_state=42)
    df = df.copy()
    df['segmento'] = km_final.fit_predict(X_sc)

    # Tabla resumen
    resumen = df.groupby('segmento')[features].agg(['mean','count'])

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, (titulo, datos_k, metric) in zip(axes, [
        ('Silhouette', ks, siluetas), ('Inercia', ks, inercias)
    ]):
        ax.plot(datos_k, metric, 'bo-', ms=7, lw=2)
        ax.axvline(k_opt, color='red', ls='--', label=f'K={k_opt}')
        ax.set_title(titulo); ax.set_xlabel('K'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

    fig2, ax2 = plt.subplots(figsize=(7, 5))
    for seg in range(k_opt):
        mask = df['segmento'] == seg
        ax2.scatter(df.loc[mask, features[0]], df.loc[mask, features[1]],
                    s=25, alpha=0.7, label=f'Segmento {seg}')
    ax2.set_xlabel(features[0]); ax2.set_ylabel(features[1])
    ax2.set_title(f'Segmentación (K={k_opt})')
    ax2.legend(); plt.tight_layout(); plt.show()

    print(f"K óptimo: {k_opt} | Silhouette: {max(siluetas):.4f}")
    return df, resumen

# Prueba con el df_clientes del cuerpo de la clase
df_cl2 = df_clientes.drop(columns=['segmento','nombre_segmento'], errors='ignore')
features = ['gasto_mensual', 'visitas_mes', 'edad']
df_seg, resumen = segmentar_clientes(df_cl2, features)
print(resumen)
```
</details>

---
## Resumen

```python
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# 1. Escalar SIEMPRE antes de KMeans
X_sc = StandardScaler().fit_transform(X)

# 2. Elegir K con Elbow + Silhouette
for k in range(2, 10):
    km = KMeans(k, n_init=15, random_state=42)
    labels = km.fit_predict(X_sc)
    print(k, km.inertia_, silhouette_score(X_sc, labels))

# 3. Entrenar el modelo final
km_final = KMeans(n_clusters=4, n_init=20, random_state=42)
labels = km_final.fit_predict(X_sc)

# 4. Atributos útiles
km_final.cluster_centers_   # centroides (en espacio escalado)
km_final.inertia_            # suma de distancias² dentro de clusters
km_final.n_iter_             # iteraciones hasta convergencia

# Regla de oro: SIEMPRE escalar → SIEMPRE usar n_init>1 → comparar Elbow + Silhouette
```

## ¿Qué viene en la próxima clase?
En la **Clase 9** aprenderemos a construir **sklearn Pipelines** completos y a guardar modelos con `joblib` para desplegarlos en producción.